# 22.2 - Buco nero di Kerr: laboratorio 3D interattivo

Questo notebook Colab si concentra soltanto sulla struttura tridimensionale vicina a un buco nero rotante. Il disco non e' un foglio: possiede spessore, turbolenza verticale, rotazione differenziale e un lieve warp. L'immagine secondaria viene ricostruita dalle particelle reali che si trovano dietro il buco nero rispetto all'osservatore.

## Cosa e' fisico e cosa e' visuale

- Kerr: orizzonte, spin e ISCO progrado seguono le formule relativistiche in unita' $r_g=GM/c^2$.
- Disco: la velocita' angolare usa $\Omega_K \propto (r^{3/2}+a_*)^{-1}$; temperatura e spessore seguono profili da disco di accrescimento.
- Doppler: il lato che si avvicina all'osservatore viene amplificato, quello che si allontana attenuato.
- Lensing: anello e immagine secondaria sono un'approssimazione geodetica costruita sul piano del cielo di un osservatore fissato. Non sostituiscono un ray tracing Kerr completo.

> Ruotando la camera si esplora la geometria fisica. Gli elementi chiamati **immagine osservata** restano riferiti all'osservatore virtuale iniziale e possono essere nascosti dalla legenda.


## 1. Preparazione Colab

Esegui questa cella una volta. Il notebook usa solo NumPy e Plotly.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26', 'plotly>=5.20',
])
print('Dipendenze installate.')


## 2. Parametri Kerr e del disco

Puoi modificare spin, inclinazione del disco esterno, numero di particelle e spessore. Valori maggiori di `N_PARTICELLE` o minori di `PASSO_FRAME` producono un HTML piu' pesante.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display

# Tutte le distanze grafiche sono espresse in raggi gravitazionali r_g.
A_STAR = 0.92
MASSA_BH_SOLARI = 6.5e9
N_PARTICELLE = 1400
N_FRAME = 241
PASSO_FRAME = 2
R_OUT = 14.0
H_OVER_R = 0.065
WARP_ESTERNO_GRADI = 11.0
SEED = 222

if not 0 <= A_STAR < 1:
    raise ValueError('A_STAR deve essere compreso tra 0 e 1 (1 escluso).')
if N_PARTICELLE < 100 or N_FRAME < 3 or PASSO_FRAME < 1:
    raise ValueError('Risoluzione della simulazione non valida.')

# Orizzonte e ISCO progrado in unita' r_g.
R_H = 1 + np.sqrt(1-A_STAR**2)
Z1 = 1 + (1-A_STAR**2)**(1/3)*((1+A_STAR)**(1/3)+(1-A_STAR)**(1/3))
Z2 = np.sqrt(3*A_STAR**2 + Z1**2)
R_ISCO = 3 + Z2 - np.sqrt((3-Z1)*(3+Z1+2*Z2))
R_IN = 1.04*R_ISCO
R_FOTONI_PROGRADO = 2*(1 + np.cos((2/3)*np.arccos(-A_STAR)))
B_CRITICO = 5.20 - 0.28*A_STAR

G = 6.67430e-11
C = 299_792_458.0
M_SOLE = 1.98847e30
R_G_METRI = G*MASSA_BH_SOLARI*M_SOLE/C**2
T_G_SECONDI = R_G_METRI/C

print(f'a*={A_STAR:.2f} | r_H={R_H:.3f} r_g | r_ISCO={R_ISCO:.3f} r_g')
print(f'r_g={R_G_METRI:.3e} m | GM/c^3={T_G_SECONDI:.1f} s')
print(f'Particelle={N_PARTICELLE} | frame WebGL={len(range(0, N_FRAME, PASSO_FRAME))}')


## 3. Disco tridimensionale

Ogni particella riceve raggio, azimut e quota. La quota e' distribuita attorno alla superficie media con una gaussiana il cui spessore cresce con il raggio. Il piano esterno e' inclinato rispetto all'equatore del buco nero: e' una rappresentazione didattica del warp e dell'allineamento di Bardeen-Petterson.


In [ ]:
rng = np.random.default_rng(SEED)

# Una miscela radiale mantiene leggibili sia il bordo interno sia il disco esterno.
u = rng.random(N_PARTICELLE)
r_area = np.sqrt(R_IN**2 + u*(R_OUT**2-R_IN**2))
r_interno = R_IN + (R_OUT-R_IN)*rng.power(.52, N_PARTICELLE)
usa_interno = rng.random(N_PARTICELLE) < .62
r0 = np.where(usa_interno, r_interno, r_area)
phi0 = rng.uniform(0, 2*np.pi, N_PARTICELLE)
quota0 = np.clip(rng.normal(0, 1, N_PARTICELLE), -2.8, 2.8)
fase_verticale = rng.uniform(0, 2*np.pi, N_PARTICELLE)
turbolenza = rng.uniform(.65, 1.35, N_PARTICELLE)

frame_sim = np.arange(N_FRAME, dtype=float)[:, None]
tempo = .18*frame_sim
deriva = .011*(R_OUT/r0[None, :])**1.25 * tempo**1.13
r3 = np.maximum(r0[None, :]-deriva, 1.015*R_H)
omega3 = 4.0/(r3**1.5 + A_STAR)
phi3 = phi0[None, :] + omega3*tempo

# Spessore H/R, oscillazione verticale e flare verso il bordo esterno.
flare3 = H_OVER_R*(.72 + .75*(r3-R_IN)/(R_OUT-R_IN))
z_locale3 = (
    quota0[None, :]*flare3*r3
    + .055*r3*np.sin(.58*phi3 + fase_verticale[None, :])*turbolenza[None, :]
)
x_locale3 = r3*np.cos(phi3)
y_locale3 = r3*np.sin(phi3)

# L'interno e' quasi equatoriale; il piano esterno conserva l'inclinazione iniziale.
frazione_warp3 = np.clip((r3-1.35*R_ISCO)/(R_OUT-1.35*R_ISCO), 0, 1)
tilt3 = np.deg2rad(WARP_ESTERNO_GRADI)*frazione_warp3**1.6
nodo3 = .35*np.log(np.maximum(r3/R_ISCO, 1))
cosn3, sinn3 = np.cos(nodo3), np.sin(nodo3)
x_nodo3 = cosn3*x_locale3 + sinn3*y_locale3
y_nodo3 = -sinn3*x_locale3 + cosn3*y_locale3
x3 = x_nodo3
y3 = y_nodo3*np.cos(tilt3) - z_locale3*np.sin(tilt3)
z3 = y_nodo3*np.sin(tilt3) + z_locale3*np.cos(tilt3)

# Temperatura relativistica semplificata e Doppler beaming per l'osservatore iniziale.
termine_bordo3 = np.maximum(1-np.sqrt(R_ISCO/r3), 1e-4)
temperatura3 = (r3/R_ISCO)**(-.75)*termine_bordo3**.25
LOS = np.array([1.25, -1.65, .78], dtype=float)
LOS /= np.linalg.norm(LOS)
vhat_x3, vhat_y3 = -np.sin(phi3), np.cos(phi3)
cos_alpha3 = vhat_x3*LOS[0] + vhat_y3*LOS[1]
beta3 = np.minimum(.68, .84/np.sqrt(r3))
gamma3 = 1/np.sqrt(1-beta3**2)
doppler3 = (1/(gamma3*(1-beta3*cos_alpha3)))**3
emissione3 = temperatura3**4*doppler3
p99 = np.percentile(emissione3, 99.4)
colore3 = np.clip(np.log1p(9*emissione3/p99)/np.log(10), 0, 1).astype(np.float32)
peso_centrale3 = .75 + .45*np.exp(-(r3-R_IN)/4.2)
dimensione3 = ((2.1 + 5.2*colore3)*peso_centrale3).astype(np.float32)
x3, y3, z3 = x3.astype(np.float32), y3.astype(np.float32), z3.astype(np.float32)

del frame_sim, deriva, omega3, x_locale3, y_locale3, x_nodo3, y_nodo3
memoria_mb = sum(a.nbytes for a in (x3, y3, z3, colore3, dimensione3))/1024**2
print(f'Disco 3D precalcolato: {memoria_mb:.1f} MB di array principali.')
print(f'Escursione verticale: {np.min(z3):.2f} ... {np.max(z3):.2f} r_g')


## 4. Anello fotonico e luce deviata

La gravita' permette alla luce del lato posteriore del disco di raggiungere l'osservatore passando sopra e sotto il buco nero. Per questo un disco quasi di taglio non appare come una semplice ellisse: compare una **immagine secondaria** avvolta attorno all'ombra. Vicino al parametro d'impatto critico, fotoni che compiono orbite quasi complete formano il sottile **anello fotonico**.

Qui questi fenomeni sono visualizzati nel piano del cielo fissato da `LOS`. L'immagine secondaria non e' un cerchio predefinito: a ogni frame deriva dalla posizione, temperatura e profondita' delle particelle sul lato posteriore. Un'immagine scientifica richiederebbe comunque ray tracing Kerr e trasferimento radiativo.


In [ ]:
# Base ortonormale del piano del cielo dell'osservatore virtuale.
E1 = np.cross(np.array([0., 0., 1.]), LOS)
E1 /= np.linalg.norm(E1)
E2 = np.cross(LOS, E1)
E2 /= np.linalg.norm(E2)

theta_ring = np.linspace(0, 2*np.pi, 520)
raggio_ring = B_CRITICO*(1 + .026*A_STAR*np.cos(theta_ring))
centro_ring = -.22*A_STAR*E1
anello3 = (
    centro_ring[None, :] + .10*LOS[None, :]
    + raggio_ring[:, None]*(np.cos(theta_ring)[:, None]*E1
                              + np.sin(theta_ring)[:, None]*E2)
)

# Disco nero sul piano del cielo: e' l'ombra apparente, non l'orizzonte materiale.
theta_ombra = np.linspace(0, 2*np.pi, 150, endpoint=False)
bordo_ombra = centro_ring[None, :] + .985*B_CRITICO*(
    np.cos(theta_ombra)[:, None]*E1 + np.sin(theta_ombra)[:, None]*E2
)
vertici_ombra3 = np.vstack([centro_ring, bordo_ombra])
i_ombra3 = np.zeros(len(theta_ombra), dtype=int)
j_ombra3 = 1 + np.arange(len(theta_ombra))
k_ombra3 = 1 + (np.arange(len(theta_ombra))+1) % len(theta_ombra)

# L'immagine secondaria verra' calcolata dalle vere particelle sul lato posteriore.
indici_lensing3 = np.arange(0, N_PARTICELLE, 2)

print('Geometria osservata pronta: ombra e anello critico; nessun cerchio materiale aggiunto.')


## 5. Esploratore WebGL

Usa Play/Pausa, trascina per orbitare e la rotellina per lo zoom. La scena parte con una vista prospettica ravvicinata; **Schermo intero** si trova in basso a destra. Dalla legenda puoi nascondere separatamente disco, ombra apparente, immagine secondaria, ISCO e anello fotonico.


In [ ]:
COLORI_DISCO = [
    [0.00, '#390006'], [0.18, '#8f0705'], [0.40, '#e3310b'],
    [0.62, '#ff8b20'], [0.80, '#ffe27a'], [0.94, '#fffbe8'], [1.00, '#d7f5ff'],
]
frame_visibili = list(range(0, N_FRAME, PASSO_FRAME))
if frame_visibili[-1] != N_FRAME-1:
    frame_visibili.append(N_FRAME-1)

def tracce_dinamiche(frame):
    # Seleziona materia realmente dietro il buco nero rispetto all'osservatore.
    idx = indici_lensing3
    punti = np.column_stack((x3[frame, idx], y3[frame, idx], z3[frame, idx]))
    profondita = punti @ LOS
    lontane = profondita < -.12
    punti = punti[lontane]
    idx = idx[lontane]
    alpha = punti @ E1
    beta = punti @ E2
    r_schermo = np.maximum(np.hypot(alpha, beta), .25)
    # Parita' invertita e compressione verso il parametro d'impatto critico.
    r_immagine = B_CRITICO + .10 + 1.15/(r_schermo+.45)
    direzione = (alpha/r_schermo)[:, None]*E1 + (beta/r_schermo)[:, None]*E2
    immagine = centro_ring[None, :] + .13*LOS[None, :] - r_immagine[:, None]*direzione
    luce_secondaria = np.clip(.24 + .92*colore3[frame, idx], 0, 1)
    return [
        go.Scatter3d(
            x=x3[frame], y=y3[frame], z=z3[frame], mode='markers', name='Disco fisico 3D',
            marker=dict(
                size=dimensione3[frame], color=colore3[frame], colorscale=COLORI_DISCO,
                cmin=0, cmax=1, opacity=.91,
                line=dict(color='rgba(255,210,135,.18)', width=.25),
            ),
            hovertemplate='Disco<br>r=%{customdata:.2f} r_g<extra></extra>',
            customdata=r3[frame],
        ),
        go.Scatter3d(
            x=immagine[:, 0], y=immagine[:, 1], z=immagine[:, 2],
            mode='markers', name='Immagine secondaria (lensing)',
            marker=dict(
                size=1.8+3.8*luce_secondaria, color=luce_secondaria,
                colorscale=COLORI_DISCO, cmin=0, cmax=1, opacity=.86,
            ),
            hoverinfo='skip',
        ),
    ]

# Superficie oblate dell'orizzonte di Kerr in coordinate Kerr-Schild visuali.
u_h = np.linspace(0, 2*np.pi, 42)
v_h = np.linspace(0, np.pi, 24)
r_eq_h = np.sqrt(R_H**2 + A_STAR**2)
xh = r_eq_h*np.outer(np.cos(u_h), np.sin(v_h))
yh = r_eq_h*np.outer(np.sin(u_h), np.sin(v_h))
zh = R_H*np.outer(np.ones_like(u_h), np.cos(v_h))

angolo_isco = np.linspace(0, 2*np.pi, 260)
rng_stelle = np.random.default_rng(SEED+1)
stelle = rng_stelle.normal(size=(220, 3))
stelle /= np.linalg.norm(stelle, axis=1)[:, None]
stelle *= rng_stelle.uniform(15.5, 21.5, len(stelle))[:, None]

statiche = [
    go.Scatter3d(
        x=stelle[:, 0], y=stelle[:, 1], z=stelle[:, 2], mode='markers',
        marker=dict(size=rng_stelle.uniform(.8, 2.7, len(stelle)), color='white', opacity=.52),
        name='Stelle', showlegend=False, hoverinfo='skip',
    ),
    go.Mesh3d(
        x=vertici_ombra3[:, 0], y=vertici_ombra3[:, 1], z=vertici_ombra3[:, 2],
        i=i_ombra3, j=j_ombra3, k=k_ombra3, name='Ombra apparente',
        color='#000006', opacity=1.0, hoverinfo='skip',
        lighting=dict(ambient=1, diffuse=0, specular=0),
    ),
    go.Surface(
        x=xh, y=yh, z=zh, name='Orizzonte Kerr', showscale=False, hoverinfo='skip',
        colorscale=[[0, '#10030b'], [.55, '#2b0715'], [1, '#7d2819']], opacity=.99,
        lighting=dict(ambient=.66, diffuse=.88, specular=.52, roughness=.68),
        lightposition=dict(x=25, y=-30, z=35), visible='legendonly',
    ),
    go.Scatter3d(
        x=R_ISCO*np.cos(angolo_isco), y=R_ISCO*np.sin(angolo_isco),
        z=np.zeros_like(angolo_isco), mode='lines', name='ISCO progrado',
        line=dict(color='#ff7b38', width=4, dash='dash'), hoverinfo='skip',
        visible='legendonly',
    ),
    go.Scatter3d(
        x=anello3[:, 0], y=anello3[:, 1], z=anello3[:, 2],
        mode='lines', name='Anello fotonico sottile',
        line=dict(color='#ffe6a1', width=2), opacity=.68,
        hoverinfo='skip',
    ),
]

iniziali = tracce_dinamiche(frame_visibili[0])
fotogrammi = [
    go.Frame(name=str(f), data=tracce_dinamiche(f), traces=[0, 1])
    for f in frame_visibili
]
passi = [
    dict(
        method='animate', label=str(f) if f % 40 == 0 or f == N_FRAME-1 else '',
        args=[[str(f)], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                             transition=dict(duration=0))],
    )
    for f in frame_visibili
]

CAMERA_INIZIALE3 = dict(
    eye=dict(x=1.38, y=-1.82, z=.72),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=1),
    projection=dict(type='perspective'),
)

fig3 = go.Figure(data=iniziali+statiche, frames=fotogrammi)
fig3.update_layout(
    title=dict(text='<b>BUCO NERO DI KERR: DISCO 3D E LUCE CURVATA</b>',
               x=.5, y=.985, xanchor='center', font=dict(size=21, color='#fff3dc')),
    paper_bgcolor='#01030a', plot_bgcolor='#01030a', height=880,
    margin=dict(l=8, r=8, t=76, b=24),
    font=dict(color='#f4eadb', family='Arial'),
    legend=dict(
        x=.01, y=.82, bgcolor='rgba(3,8,18,.72)', bordercolor='#74401f', borderwidth=1,
        font=dict(size=11), itemsizing='constant',
    ),
    modebar=dict(orientation='v', bgcolor='rgba(24,12,5,.9)',
                 color='#e6d5bc', activecolor='#ff9d2e'),
    uirevision='camera-3d-kerr',
    scene=dict(
        bgcolor='#01030a',
        xaxis=dict(visible=False, range=[-18, 18]),
        yaxis=dict(visible=False, range=[-18, 18]),
        zaxis=dict(visible=False, range=[-12, 12]),
        aspectmode='data',
        camera=CAMERA_INIZIALE3,
    ),
    updatemenus=[dict(
        type='buttons', direction='left', x=.015, y=.985,
        xanchor='left', yanchor='top', showactive=False,
        bgcolor='rgba(24,12,5,.92)', bordercolor='#ff9d2e', borderwidth=1,
        font=dict(size=14, color='#fff4dc'), pad=dict(l=7, r=7, t=7, b=7),
        buttons=[
            dict(label='&#9654;&nbsp; Play', method='animate',
                 args=[None, dict(fromcurrent=True, mode='immediate',
                                      frame=dict(duration=70, redraw=True),
                                      transition=dict(duration=0))]),
            dict(label='&#10074;&#10074;&nbsp; Pausa', method='animate',
                 args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False),
                                         transition=dict(duration=0))]),
        ],
    )],
    sliders=[dict(
        active=0, steps=passi, x=.10, len=.80,
        bgcolor='rgba(20,10,4,.76)', bordercolor='#7f4a1f',
        currentvalue=dict(prefix='Frame ', font=dict(color='#ffe4b5', size=12)),
        transition=dict(duration=0),
    )],
)

CONFIG3 = {
    'scrollZoom': True, 'responsive': True, 'displayModeBar': True, 'displaylogo': False,
    'modeBarButtonsToRemove': ['hoverClosest3d', 'toggleSpikelines'],
    'toImageButtonOptions': {
        'format': 'png', 'filename': '022_2_buco_nero_kerr_3d',
        'height': 1000, 'width': 1500, 'scale': 1,
    },
}
SCRIPT_FULLSCREEN3 = r'''
const grafico = document.getElementById('{plot_id}');
grafico.style.position = 'relative';
const pulsante = document.createElement('button');
pulsante.textContent = 'Schermo intero';
pulsante.style.cssText = 'position:absolute;right:18px;bottom:18px;z-index:30;padding:11px 16px;' +
  'border:1px solid #ff9d2e;border-radius:5px;background:rgba(24,12,5,.94);' +
  'color:#fff4dc;font:600 13px Arial;cursor:pointer';
pulsante.onclick = () => {
  if (!document.fullscreenElement) {
    (grafico.requestFullscreen || grafico.webkitRequestFullscreen).call(grafico);
  } else {
    (document.exitFullscreen || document.webkitExitFullscreen).call(document);
  }
};
document.addEventListener('fullscreenchange', () => {
  const attivo = document.fullscreenElement === grafico;
  grafico.style.width = attivo ? '100vw' : '100%';
  grafico.style.height = attivo ? '100vh' : '880px';
  grafico.style.background = '#01030a';
  pulsante.textContent = attivo ? 'Esci da schermo intero' : 'Schermo intero';
  Plotly.Plots.resize(grafico);
});
grafico.appendChild(pulsante);
'''

PERCORSO_HTML3 = '/content/022_2_buco_nero_3d_interattivo.html'
fig3.write_html(
    PERCORSO_HTML3, include_plotlyjs=True, full_html=True, auto_play=False,
    config=CONFIG3, post_script=SCRIPT_FULLSCREEN3,
)
html3 = fig3.to_html(
    include_plotlyjs='cdn', full_html=False, auto_play=False,
    config=CONFIG3, post_script=SCRIPT_FULLSCREEN3,
)
# Il rendering diretto evita i limiti di dimensione di srcdoc con animazioni grandi.
display(HTML(html3))
print(f'Esploratore salvato in: {PERCORSO_HTML3}')


## 6. Download dell'esploratore

Il file contiene Plotly e funziona anche fuori da Colab. Conserva animazione, legenda, rotazione, zoom e schermo intero.


In [ ]:
from google.colab import files
files.download(PERCORSO_HTML3)


## 7. Come interpretare la scena

- **Disco fisico 3D:** le particelle occupano un volume attorno alla superficie media; non sono distribuite casualmente in una sfera. Il momento angolare mantiene una struttura toroidale appiattita.
- **Warp:** il disco interno tende all'equatore del buco nero, quello esterno conserva una piccola inclinazione.
- **Asimmetria luminosa:** e' principalmente Doppler beaming. Il lato che viene verso l'osservatore appare piu' brillante.
- **Anello fotonico:** e' l'immagine di fotoni con parametro d'impatto vicino al valore critico, non un anello materiale.
- **Immagine secondaria:** rappresenta luce del lato posteriore deviata sopra e sotto l'ombra. Non coincide con le particelle fisiche del disco.
- **Limite:** l'evoluzione del disco e' cinematica e il lensing e' parametrico. Per ottenere pixel scientifici servono geodetiche nulle di Kerr, redshift gravitazionale, assorbimento e trasferimento radiativo.

### Esperimenti

Prova `A_STAR=0`, `A_STAR=0.98`, `WARP_ESTERNO_GRADI=0` oppure aumenta `H_OVER_R`. Dopo una modifica riesegui dalla cella dei parametri in avanti.
